# Cardiac Abnormality Assessment with KaleCardiac

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pykale/cardiac/blob/main/tutorials/cardiac-abnormality-assessment/tutorial-CardioVAE.ipynb)

In this tutorial, we use **KaleCardiac** to pretrain, fine-tune, evaluate and interpret **CardioVAE** [1], a
multimodal variational autoencoder (VAE) over two low-cost, non-invasive cardiac investigations: a **chest X-ray
(CXR)** and a **twelve-lead electrocardiogram (ECG)**. The fine-tuned model classifies each subject as **healthy** or
as having a **cardiothoracic abnormality**.

CardioVAE learns its representation without labels, from paired ECGs and CXRs, with a *tri-stream* objective: one
evidence lower bound (ELBO) through both modalities together, and one through each modality alone, so that each
encoder stays useful when the other modality is missing. The two modalities are fused at the feature-embedding level
(intermediate fusion). A small labelled cohort then fine-tunes a classifier on top of the pretrained encoders.

The workflow starts from **raw DICOM files**, the format hospitals export both investigations in, and each step is a
few calls to the KaleCardiac API:

- **Step 0: Environment preparation** -- install KaleCardiac, download the tutorial dataset, and load the configuration.
- **Step 1: Multimodal pretraining** -- load the unlabelled ECG-CXR pairs, prepare them, and pretrain CardioVAE.
- **Step 2: Multimodal fine-tuning** -- load the labelled pairs, prepare them, and fine-tune a classifier.
- **Step 3: Evaluation** -- score the held-out subjects.
- **Step 4: Interpretation** -- find what in each ECG and CXR the predictions rested on.

**The tutorial dataset** holds 20 subjects, each with a twelve-lead ECG and a chest X-ray stored as DICOM, in two
copies: `kalecardiac_pretraining`, without labels, and `kalecardiac_finetuning`, with a label file. It exists to
show the workflow end to end: the 20 subjects are built from a handful of recordings, the same subjects serve
pretraining and fine-tuning, and the labels are illustrative rather than clinical findings. **The numbers this
notebook prints show that the workflow runs; they do not measure CardioVAE's performance.**

**Estimated runtime:** about 10 minutes on a Colab T4 GPU, installation included.

## Step 0: Environment Preparation

### Google Colab runtime

To run this tutorial in Google Colab, use a GPU runtime: open **Runtime > Change runtime type** and choose
**T4 GPU**. The tutorial also runs on a CPU, more slowly.

### Package installation

The cell below clones the KaleCardiac repository -- which holds, besides the package, the
[CardioVAE example](https://github.com/pykale/cardiac/tree/main/examples/cardiovae_multimodal) whose configurations this tutorial reads -- and installs the package with two
optional extras: `imaging` (pydicom and scikit-image, which PyKale's DICOM reader needs) and `interpret` (Captum, for
integrated gradients). Running locally from a clone of the repository, skip it, and install with
`pip install -e ".[imaging,interpret]"` instead.

**Estimated runtime:** 3 minutes

In [ ]:
%%capture
!git clone --quiet --single-branch --branch main https://github.com/pykale/cardiac
!pip install --quiet "./cardiac[imaging,interpret]"

### Setup

The tutorial runs from the root of the KaleCardiac repository, where the CardioVAE example and its configurations
are: in Colab, the clone made above; locally, two folders above this notebook. Warnings and PyTorch Lightning's
start-up messages are silenced, so the output shows what matters.

In [ ]:
import logging
import os
import warnings
from pathlib import Path

import lightning_fabric
import pytorch_lightning

if Path("cardiac", "examples").is_dir():
    os.chdir("cardiac")
elif Path.cwd().name == "cardiac-abnormality-assessment":
    os.chdir(Path("..", ".."))
print(f"working in {Path.cwd()}")

warnings.filterwarnings("ignore")
# Lightning sets its loggers' level when it is imported, so they are quieted after it.
for module in (pytorch_lightning, lightning_fabric):
    logging.getLogger(module.__name__).setLevel(logging.ERROR)

### Dataset download

The tutorial dataset is one ZIP archive on Dropbox. The cell below downloads it into the runtime's local storage --
no Google Drive, and no sign-in -- and unpacks it into `data/`, in the working directory the setup chose:

```
data/
├── kalecardiac_pretraining/
│   ├── ECG/<subject>/.../*.dcm    one twelve-lead ECG per subject (DICOM ECG waveform)
│   └── CXR/<subject>/.../*.dcm    one chest X-ray per subject (DICOM radiograph)
└── kalecardiac_finetuning/
    ├── ECG/<subject>/.../*.dcm
    ├── CXR/<subject>/.../*.dcm
    └── labels.csv                 subject_id,label
```

Each subject is a folder named by the subject's identifier, holding the subject's DICOM file, as a hospital export
lays them out. A subject's ECG and CXR are paired by that name. The archive holds the data alone: the code and the
configurations come with KaleCardiac. To use your own data instead, skip this cell, and see *Your own data* below.

**Estimated runtime:** 1 minute

In [ ]:
import urllib.request
import zipfile

# The tutorial dataset on Dropbox; "dl=1" asks Dropbox for the file itself rather than its web page.
DATA_URL = "https://www.dropbox.com/scl/fi/y488iouenrk641dxtd8ai/kalecardiac_tutorial_data.zip?rlkey=rorf5mvbt24jdw2cr3ecyy91n&st=yt83r37g&dl=1"

if not Path("data").is_dir():
    archive, _ = urllib.request.urlretrieve(DATA_URL, "kalecardiac_tutorial_data.zip")
    with zipfile.ZipFile(archive) as bundle:
        bundle.extractall("data")
print(sorted(folder.name for folder in Path("data").iterdir()))

### Configuration

KaleCardiac reads every setting from a [YACS](https://github.com/rbgirshick/yacs) configuration, which keeps an
experiment's settings apart from its code. This tutorial trains the CardioVAE example's model, so its two
configurations live with that example, in `examples/cardiovae_multimodal/configs/`, and each is merged onto the
example's defaults -- `examples/cardiovae_multimodal/config.py`, which extends KaleCardiac's own:

- `tutorial_pretrain.yaml` -- the unlabelled dataset, and pretraining's latent width and schedule.
- `tutorial_finetune.yaml` -- the labelled dataset, the endpoint its labels record, the train/validation/test split
  and the fine-tuning schedule.

The rest is the example's: the ECG and CXR shape CardioVAE reads, the order its leads are taken in, its architecture,
and the weights of the tri-stream objective. Change a setting in either file, or override it in the cells below.
With the dataset somewhere other than `data/`, point `DATASET.ROOT` at it.

In [ ]:
from examples.cardiovae_multimodal.config import CONFIG_DIR, get_cfg_defaults

cfg_pt = get_cfg_defaults()
cfg_pt.merge_from_file(CONFIG_DIR / "tutorial_pretrain.yaml")

# ------ Hyperparameters to play with -----
cfg_pt.MODEL.LATENT_DIM = 128
cfg_pt.SOLVER.MAX_EPOCHS = 10
cfg_pt.SOLVER.BATCH_SIZE = 4
cfg_pt.OBJECTIVE.RECONSTRUCTION_WEIGHTS = ["ecg:10", "cxr:1"]
cfg_pt.OBJECTIVE.ANNEALING_EPOCHS = cfg_pt.SOLVER.MAX_EPOCHS  # the KL weight rises over the whole run

print(f"DATASET.ROOT: {cfg_pt.DATASET.ROOT}\n")
for section in ("ECG", "IMAGE", "OBJECTIVE", "SOLVER"):
    print(f"{section}:\n{cfg_pt[section]}\n")

In [ ]:
cfg_ft = get_cfg_defaults()
cfg_ft.merge_from_file(CONFIG_DIR / "tutorial_finetune.yaml")

# ------ Hyperparameters to play with -----
cfg_ft.FINETUNE.MAX_EPOCHS = 20
cfg_ft.FINETUNE.BATCH_SIZE = 4
cfg_ft.MODEL.HEAD_HIDDEN = [128]

print(f"DATASET.ROOT: {cfg_ft.DATASET.ROOT}")
print(f"ENDPOINT: {cfg_ft.ENDPOINT.NAME}, the {cfg_ft.ENDPOINT.LABEL_COLUMN!r} column of labels.csv\n")
print(f"FINETUNE:\n{cfg_ft.FINETUNE}")

### Your own data

The same steps run on your own ECGs and chest X-rays, laid out as the tutorial dataset is: `ECG/` and `CXR/`, each
holding one folder per subject with that subject's DICOM file, and, for fine-tuning, `labels.csv` with a
`subject_id` column and a column of classes (0 or 1). Uncomment the cell below and point it at your folders; the
dataset download above can then be skipped. Your ECGs' sampling rate is read from their DICOM headers, and they are
resampled to the 500 Hz CardioVAE reads.

In [ ]:
# cfg_pt.DATASET.ROOT = "/path/to/your/pretraining_data"  # holds ECG/ and CXR/
# cfg_ft.DATASET.ROOT = "/path/to/your/finetuning_data"  # holds ECG/, CXR/ and labels.csv
# cfg_ft.ENDPOINT.LABEL_COLUMN = "label"  # the column of labels.csv holding each subject's class

## Step 1: Multimodal Pretraining

CardioVAE is pretrained without labels, on the paired ECGs and CXRs of `kalecardiac_pretraining`.

### Data loading

Each modality is a folder holding one folder per subject, with the subject's DICOM file beneath it.
`ECGFileSource.from_dicom` and `ImageFileSource.from_dicom`, in `kalecardiac.loaddata`, index those folders by
subject and read a subject's file when it is needed, so a cohort of any size loads the same way. Each file is read
with PyKale's [3] DICOM reader,
[`kale.loaddata.image_access.read_dicom_phases`](https://github.com/pykale/pykale/blob/main/kale/loaddata/image_access.py),
and then:

- **ECG:** the twelve-lead rhythm waveform is taken out of the DICOM ECG in millivolts, with each lead's name and the
  sampling rate read from the file's header. Every other subject's leads are found by *name*, so a file that stores
  aVF before aVL cannot transpose them.
- **CXR:** the radiograph's pixels are mapped through the display settings its header carries -- the rescale, the
  window, and the inversion a `MONOCHROME1` image needs -- so the model sees the image a clinician sees.

**Estimated runtime:** 10 seconds

In [ ]:
from kalecardiac.loaddata import ECGFileSource, ImageFileSource

ecg_dir_pt, cxr_dir_pt = Path(cfg_pt.DATASET.ROOT, "ECG"), Path(cfg_pt.DATASET.ROOT, "CXR")
ecg_raw = ECGFileSource.from_dicom(ecg_dir_pt)
cxr_raw = ImageFileSource.from_dicom(cxr_dir_pt)
print(ecg_raw, cxr_raw, sep="\n")

One subject's ECG and CXR, as loaded: lead II of the full recording, and the radiograph at its own resolution.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

subject = sorted(ecg_raw.paths)[0]
recording, radiograph = ecg_raw.get(subject, 0).numpy(), cxr_raw.get(subject, 0).numpy()[0]

fig, (ax_ecg, ax_cxr) = plt.subplots(1, 2, figsize=(15, 4), gridspec_kw={"width_ratios": [3, 1]})
time = np.arange(recording.shape[1]) / ecg_raw.sampling_rate
ax_ecg.plot(time, recording[ecg_raw.leads.index("II")], color="black", linewidth=0.8)
ax_ecg.set(xlabel="Time (seconds)", ylabel="mV", title=f"{subject}: ECG lead II, as recorded")
ax_cxr.imshow(radiograph, cmap="gray")
ax_cxr.set_title(f"{subject}: CXR, {radiograph.shape[0]} x {radiograph.shape[1]} pixels")
ax_cxr.axis("off")
plt.tight_layout()
plt.show()

### Data preparation

CardioVAE reads a ten-second, 500 Hz twelve-lead ECG with its leads interleaved, sample by sample, into one
60,000-sample strip, and a 224 x 224 CXR scaled to [0, 1]. Two KaleCardiac transforms build those from the
configuration:

- `get_ecg_transform_from_cfg` (`kalecardiac.auto`) reads `cfg.ECG`: fill any dropped samples, resample if the
  recording arrives at another rate, crop or pad each lead to ten seconds, normalize each lead to zero mean and unit
  variance, then interleave the leads. Every step before the interleaving works on one lead at a time, so none mixes
  one lead into the next. The leads are taken by name in `ECG.LEADS`' order -- the order the published CardioVAE
  was pretrained in, aVF before aVL -- whichever order a file stores them in.
- `get_image_transform` (`kalecardiac.prepdata`) resizes the CXR, with anti-aliasing, and scales it to [0, 1].

The sources apply them as each subject is read. `MultimodalDataset` then pairs the two modalities by subject: each
subject yields its ECG as `ecg` and its CXR as `cxr`, the names the objective's weights and the model's encoders use.

**Estimated runtime:** 10 seconds

In [ ]:
from kalecardiac.auto import get_ecg_transform_from_cfg
from kalecardiac.loaddata import MultimodalDataset
from kalecardiac.prepdata import get_image_transform

# The rate the recordings were made at, as their DICOM headers state it, so any other is resampled.
cfg_pt.ECG.SOURCE_SAMPLING_RATE = ecg_raw.sampling_rate
ecg_transform = get_ecg_transform_from_cfg(cfg_pt)
cxr_transform = get_image_transform(size=cfg_pt.IMAGE.SIZE)

ecg = ECGFileSource.from_dicom(ecg_dir_pt, select=cfg_pt.ECG.LEADS, transform=ecg_transform)
cxr = ImageFileSource.from_dicom(cxr_dir_pt, transform=cxr_transform)
pretrain_set = MultimodalDataset(sorted(ecg.paths.keys() & cxr.paths.keys()), {"ecg": ecg, "cxr": cxr})

print(pretrain_set)
print({name: tuple(value.shape) for name, value in pretrain_set[0].modalities.items()})

### Multimodal pretraining

`CardioVAE`, in `kalecardiac.model.embed`, has an encoder and a decoder per modality -- a 1-D convolutional network
for the ECG strip and a 2-D one for the CXR -- and fuses the two with a **product of experts**: each encoder's
Gaussian over the shared latent space, and a standard-normal prior, multiply into one joint posterior. A subject
missing a modality contributes no expert. Its input sizes come from the prepared data.

`pretrain`, in `kalecardiac.auto`, trains it with the configured objective -- the tri-stream ELBO, the ECG's
reconstruction weighted ten times the CXR's and the KL term annealed in -- and saves the weights to
`pretrained.pt` in `OUTPUT.OUT_DIR`. It needs no labels: pretraining is a reconstruction task.

**Estimated runtime:** 1 to 2 minutes

In [ ]:
import pandas as pd

from kalecardiac.auto import pretrain
from kalecardiac.model.embed import CardioVAE
from kalecardiac.utils import set_seed

set_seed(cfg_pt.SOLVER.SEED)
model = CardioVAE(
    signal_length=ecg.length,
    image_size=cxr.size,
    latent_dim=cfg_pt.MODEL.LATENT_DIM,
    signal_name="ecg",
    image_name="cxr",
)
print(f"CardioVAE with {sum(parameter.numel() for parameter in model.parameters()):,} parameters")

pretrained = pretrain(cfg_pt, model, pretrain_set, cfg_pt.OUTPUT.OUT_DIR)

# The training loss per epoch, from the run's log.
log = max(Path(cfg_pt.OUTPUT.OUT_DIR).glob("pretrain/history/*/metrics.csv"), key=lambda path: path.stat().st_mtime)
pd.read_csv(log).groupby("epoch")["train_loss"].mean().round(1).to_frame("training loss").T

## Step 2: Multimodal Fine-tuning

The pretrained encoders are fine-tuned on the labelled pairs of `kalecardiac_finetuning` to classify each subject:
**0** healthy, **1** cardiothoracic abnormality.

### Data loading

The labelled cohort has the pretraining layout plus `labels.csv`, one row per subject. Its ECGs and CXRs load
through the same DICOM sources, with the same transforms -- the pretrained encoders read exactly that shape -- and
the labels are an ordinary table.

**Estimated runtime:** 1 second

In [ ]:
ecg_ft = ECGFileSource.from_dicom(Path(cfg_ft.DATASET.ROOT, "ECG"), select=cfg_pt.ECG.LEADS, transform=ecg_transform)
cxr_ft = ImageFileSource.from_dicom(Path(cfg_ft.DATASET.ROOT, "CXR"), transform=cxr_transform)
labels = pd.read_csv(Path(cfg_ft.DATASET.ROOT, "labels.csv"))

print(ecg_ft, cxr_ft, sep="\n")
labels.head()

### Data preparation

Three calls turn the labels into a fine-tuning run, each reading `cfg_ft`:

- `derive_endpoint` reads each subject's class from `ENDPOINT.LABEL_COLUMN`, dropping a subject with none rather than
  assuming it negative, and `get_endpoint_target` makes the class the supervision every batch carries.
- `get_folds` divides the subjects into training, validation and test sets, stratified by label, with no subject in
  two of them: 20% are held out for testing, and the validation set that chooses the checkpoint is carved out of the
  rest -- never out of the test set.

The class weight the loss uses is taken from the training set alone, when the fine-tuning task is built.

**Estimated runtime:** 1 second

In [ ]:
from kalecardiac.auto import derive_endpoint, get_endpoint_target, get_folds

table = derive_endpoint(cfg_ft, labels)
target = get_endpoint_target(cfg_ft, table, id_column="subject_id")
fold = get_folds(cfg_ft, table)[0]
print(
    f"{len(fold.train_ids)} subjects to train on, {len(fold.val_ids)} to validate on, {len(fold.test_ids)} to test on"
)

### Multimodal fine-tuning

`fine_tune`, in `kalecardiac.auto`, keeps the pretrained CardioVAE's encoders, frozen, as feature extractors, and
trains a classification head on the posterior means of both modalities, concatenated. The head, the loss and the
positive class's weight follow from the endpoint and the training labels. After training, the checkpoint that scored
best on the validation subjects is restored, and the test subjects are scored.

**Estimated runtime:** 1 to 2 minutes

In [ ]:
from kalecardiac.auto import fine_tune

set_seed(cfg_ft.SOLVER.SEED)
result = fine_tune(cfg_ft, pretrained, {"ecg": ecg_ft, "cxr": cxr_ft}, target, fold, cfg_ft.OUTPUT.OUT_DIR)

## Step 3: Evaluation

`fine_tune` has scored the held-out test subjects with `score_predictions`, which chooses the metrics an endpoint
calls for -- for a binary one, `binary_metrics` in `kalecardiac.evaluate`:

- **Accuracy:** the share of subjects classified correctly.
- **AUROC:** the area under the receiver operating characteristic curve -- how well the scores rank positive subjects
  above negative ones, at every threshold at once.
- **MCC:** the Matthews correlation coefficient, which stays informative when the classes are imbalanced.
- **Sensitivity** and **specificity:** the shares of positive and of negative subjects classified correctly.

The predictions are kept per subject, so they can be inspected, or scored again at another threshold. With only a
few test subjects, the scores can rank them correctly -- a high AUROC -- and still all fall on one side of the 0.5
threshold that accuracy, MCC, sensitivity and specificity cut at. On this dataset -- 20 subjects built from a
handful of recordings -- the numbers show that the evaluation runs, not how well CardioVAE performs.

In [ ]:
metrics = pd.Series(result.metrics)[["accuracy", "roc_auc", "mcc", "sensitivity", "specificity"]].astype(float)
metrics.rename({"roc_auc": "AUROC", "mcc": "MCC"}).round(3).to_frame("test")

In [ ]:
predictions = pd.DataFrame(result.predictions.rows())
predictions.assign(probability=1 / (1 + np.exp(-predictions["score"]))).round(3)

## Step 4: Interpretation

Two questions, both answered from the fine-tuned model:

- **Which modality did it rely on?** `interpret_model`, in `kalecardiac.auto`, runs integrated gradients over the
  test subjects and reports each modality's *attribution ratio* -- the share of its samples or pixels that were
  strongly attributed. It then withholds each modality in turn, through the presence mask, as if no subject had it,
  and scores the test set again: a modality whose loss costs nothing is one the model could do without.
- **Where in each modality?** `modality_attributions`, in `kalecardiac.interpret`, attributes one subject's prediction
  to every sample of its ECG and every pixel of its CXR with [Captum](https://captum.ai)'s integrated gradients [2],
  and `normalize_attribution` rescales each map to [0, 1], so the regions above a threshold can be drawn.

**Estimated runtime:** 10 seconds

In [ ]:
from kalecardiac.auto import interpret_model

summary = interpret_model(cfg_ft, result.model, result.test_loader, cfg_ft.OUTPUT.OUT_DIR)
pd.DataFrame(summary["attribution"]).T.round(3)

The test AUROC with every modality, and with each one withheld:

In [ ]:
pd.Series(summary.get("ablation", {}), name="test AUROC").round(3).to_frame()

One test subject's prediction, attributed to its ECG and CXR. The strip interleaves the twelve leads, so one lead is
read out of it by name.

In [ ]:
import torch

from kalecardiac.interpret import modality_attributions, normalize_attribution
from kalecardiac.loaddata import collate_subjects

# Change these to explain another test subject or lead, or to move the thresholds.
SAMPLE_IDX, LEAD = 0, "II"
ECG_THRESHOLD, CXR_THRESHOLD, ZOOM_RANGE = 0.75, 0.75, (3.5, 4.0)

sample = collate_subjects([result.test_loader.dataset[SAMPLE_IDX]])
attributions = modality_attributions(result.model.model, sample.modalities, present=sample.present)

# The strip holds the leads sample by sample: one row per sample, one column per lead.
lead, num_leads = ecg_ft.leads.index(LEAD), len(ecg_ft.leads)
ecg_signal = sample.modalities["ecg"][0, 0].numpy().reshape(-1, num_leads)[:, lead]
ecg_importance = normalize_attribution(attributions["ecg"][0, 0]).reshape(-1, num_leads)[:, lead]
ecg_time = np.arange(ecg_signal.size) / cfg_ft.ECG.SAMPLING_RATE
cxr_image = sample.modalities["cxr"][0, 0].numpy()
cxr_importance = normalize_attribution(attributions["cxr"][0, 0])

probability = torch.sigmoid(result.model.predict(sample).prediction).item()
print(
    f"{sample.subject_id[0]}: label {int(sample.target['label'])}, predicted probability of a cardiothoracic "
    f"abnormality {probability:.2f}"
)

**Full ECG view:** the whole recording of one lead, with the samples whose attribution clears the threshold drawn
in red.

In [ ]:
def plot_ecg(window: slice, title: str, width: float = 1.0):
    fig, ax = plt.subplots(figsize=(12, 4))
    ax.plot(ecg_time[window], ecg_signal[window], color="black", linewidth=width)
    for index in np.flatnonzero(ecg_importance[window] >= ECG_THRESHOLD):
        stretch = slice(max(index - 2, 0), index + 3)
        ax.plot(ecg_time[window][stretch], ecg_signal[window][stretch], color="red", linewidth=2 * width)
    ax.set_xlabel("Time (seconds)", fontsize="x-large")
    ax.set_title(title, fontsize="x-large")
    ax.legend(["ECG waveform", "Important regions"], fontsize="medium")
    plt.tight_layout()
    plt.show()


plot_ecg(slice(None), f"Full ECG, lead {LEAD}, with important regions")

**Zoomed-in ECG segment:** a short stretch of the same lead -- 3.5 to 4 seconds by default -- in finer detail.

In [ ]:
first, last = ZOOM_RANGE
window = slice(int(first * cfg_ft.ECG.SAMPLING_RATE), int(last * cfg_ft.ECG.SAMPLING_RATE))
plot_ecg(window, f"Zoomed-in ECG, lead {LEAD} ({first:.2f} s - {last:.2f} s)", width=3)

**CXR attribution map:** the pixels whose attribution clears the threshold, marked on the radiograph and coloured by
their importance.

In [ ]:
from scipy.ndimage import binary_dilation

y_points, x_points = np.where(binary_dilation(cxr_importance >= CXR_THRESHOLD, iterations=1))

fig, ax = plt.subplots(figsize=(6, 6))
ax.imshow(cxr_image, cmap="gray")
points = ax.scatter(
    x_points,
    y_points,
    c=cxr_importance[y_points, x_points],
    cmap="summer",
    marker="x",
    s=150,
    vmin=CXR_THRESHOLD,
    vmax=1.0,
    alpha=0.8,
    linewidths=0.7,
)
plt.colorbar(points, ax=ax, fraction=0.04, pad=0.04).set_label(f"Importance ({CXR_THRESHOLD:.2f}-1.00)")
ax.axis("off")
ax.set_title("CXR: important regions", fontsize="x-large")
plt.tight_layout()
plt.show()

## Where next

* The [CardioVAE example](https://github.com/pykale/cardiac/tree/main/examples/cardiovae_multimodal) runs this model from the command line, with cross-validation, modality
  ablation and a written report: `python -m examples.cardiovae_multimodal.main --cfg <your config>`. Its
  `configs/` folder holds this tutorial's two configurations beside its own.
* The [LS-EMVAE example](https://github.com/pykale/cardiac/tree/main/examples/lsemvae_ecg) treats each ECG lead as a
  modality of its own.
* The [API overview](https://github.com/pykale/cardiac/blob/main/API_overview.md) lists every stage of the
  KaleCardiac API.

## References

[1] Suvon, M. N., Tripathi, P. C., Fan, W., Zhou, S., Liu, X., Alabed, S., ... & Lu, H. (2024). Multimodal variational
autoencoder for low-cost cardiac hemodynamics instability detection. In *International Conference on Medical Image
Computing and Computer-Assisted Intervention* (pp. 296-306).

[2] Sundararajan, M., Taly, A., & Yan, Q. (2017). Axiomatic attribution for deep networks. In *International
Conference on Machine Learning* (pp. 3319-3328).

[3] Lu, H., Liu, X., Zhou, S., Turner, R., Bai, P., Koot, R. E., ... & Xu, H. (2022). PyKale: Knowledge-aware machine
learning from multiple sources in Python. In *Proceedings of the 31st ACM International Conference on Information &
Knowledge Management* (pp. 4274-4278).